# 04 — Direct vs Storm-Conditioned Hail Modeling

## Purpose

Notebook 03 constructed storm occurrence independently of the NOAA hail outcome:

$$
\text{MRMS radar}
\rightarrow
S
\rightarrow
\text{NOAA hail overlay}
\rightarrow
H.
$$

This notebook uses that storm-first sample frame for a **retrospective modeling diagnostic**.

The central question is:

> Does restricting the training sample to radar-defined storms improve the ability of coarse environmental predictors to distinguish hail-producing storms from non-hail storms?

Two formulations are compared.

### Direct formulation

$$
P(H=1 \mid X)
$$

is fitted using all radar-coverage-eligible grid-hours.

### Storm-conditioned formulation

$$
P(H=1 \mid S=1, X)
$$

is fitted using only radar-defined storm grid-hours.

Both formulations are evaluated on the **same held-out storm population**, so differences in test-set prevalence do not drive the comparison.

This is not yet a forecasting experiment.

Storm and hail labels refer to the same retrospective analysis hour, and the environmental predictors are aligned to that hour. A later notebook introduces explicit forecast origins, future storm and hail windows, and the hierarchical factorization

$$
P(H^+=1 \mid X^-)
=
P(S^+=1 \mid X^-)
P(H^+=1 \mid S^+=1, X^-).
$$

## 1. Experimental design

The observational unit is one 0.25° grid cell within one validation period.

For the retrospective pilot:

- $S=1$ means the frozen strict MRMS storm rule from Notebook 03 is satisfied;
- $H=1$ means at least one NOAA hail report is observed in the grid cell;
- $X$ contains ERA5 environmental predictors aligned to the same analysis hour.

The direct and storm-conditioned formulations differ only in their **training populations**.

The direct model is trained on:

$$
S=0 \text{ and } S=1
$$

coverage-eligible grid-hours.

The storm-conditioned model is trained on:

$$
S=1
$$

grid-hours only.

For a fair diagnostic comparison, both models are tested only on held-out $S=1$ grid-hours.

Validation is performed by leaving out one entire analysis period at a time.

This period-level split is preferable to random row splitting because grid cells within the same meteorological period are not independent observations.

The pilot contains only three hail-active periods and a small number of hail-positive storm cells. Results are therefore interpreted as diagnostic evidence rather than population-level performance estimates.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
    log_loss,
)


# ------------------------------------------------------------
# Locate repository root
# ------------------------------------------------------------

cwd = Path.cwd()

if (
    cwd
    / "data"
).exists():

    REPO_ROOT = cwd

elif (
    cwd.parent
    / "data"
).exists():

    REPO_ROOT = cwd.parent

else:

    raise FileNotFoundError(
        "Could not locate repository root. "
        "Run this notebook from the repository root "
        "or notebooks/."
    )


DATA_DIR = (
    REPO_ROOT
    / "data"
)

OUTPUT_DIR = (
    REPO_ROOT
    / "outputs"
)

TABLE_DIR = (
    OUTPUT_DIR
    / "tables"
)


print(
    "Repository root located."
)

Repository root located.


## 2. Load the frozen storm-first outputs

Notebook 03 saved compact derived tables for downstream modeling.

The main interface used here is:

```text
outputs/tables/storm_first_grid.csv

In [2]:
STORM_GRID_PATH = (
    TABLE_DIR
    / "storm_first_grid.csv"
)

STORM_SAMPLE_PATH = (
    TABLE_DIR
    / "storm_conditioned_sample.csv"
)


if not STORM_GRID_PATH.exists():

    raise FileNotFoundError(
        f"Missing Notebook 03 output: "
        f"{STORM_GRID_PATH.name}"
    )


if not STORM_SAMPLE_PATH.exists():

    raise FileNotFoundError(
        f"Missing Notebook 03 output: "
        f"{STORM_SAMPLE_PATH.name}"
    )


storm_hail_grid = pd.read_csv(
    STORM_GRID_PATH
)


storm_conditioned_sample = pd.read_csv(
    STORM_SAMPLE_PATH
)


# ------------------------------------------------------------
# Normalize saved boolean columns
# ------------------------------------------------------------

def coerce_bool_column(
    series,
):
    """
    Convert a CSV-loaded boolean column robustly to bool.
    """

    if pd.api.types.is_bool_dtype(
        series
    ):

        return series


    normalized = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )


    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    }


    converted = (
        normalized
        .map(
            mapping
        )
    )


    if converted.isna().any():

        bad_values = (
            series[
                converted.isna()
            ]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            "Could not parse boolean values: "
            f"{bad_values}"
        )


    return converted.astype(bool)


for column in [
    "coverage_eligible",
    "storm_indicator",
    "hail_positive",
]:

    storm_hail_grid[
        column
    ] = coerce_bool_column(
        storm_hail_grid[
            column
        ]
    )


print(
    "Full storm-first grid:",
    storm_hail_grid.shape,
)

print(
    "Storm-conditioned sample:",
    storm_conditioned_sample.shape,
)

Full storm-first grid: (504, 19)
Storm-conditioned sample: (123, 21)


In [3]:
# ------------------------------------------------------------
# Notebook 03 interface audit
# ------------------------------------------------------------

n_total = len(
    storm_hail_grid
)


n_eligible = int(
    storm_hail_grid[
        "coverage_eligible"
    ]
    .sum()
)


n_storm = int(
    (
        storm_hail_grid[
            "coverage_eligible"
        ]
        &
        storm_hail_grid[
            "storm_indicator"
        ]
    )
    .sum()
)


n_hail_storm = int(
    (
        storm_hail_grid[
            "coverage_eligible"
        ]
        &
        storm_hail_grid[
            "storm_indicator"
        ]
        &
        storm_hail_grid[
            "hail_positive"
        ]
    )
    .sum()
)


interface_audit = pd.DataFrame(
    {
        "quantity": [
            "Total grid-hours",
            "Coverage-eligible grid-hours",
            "Strict storm grid-hours",
            "Observed hail storms",
            "Primary periods",
        ],

        "value": [
            n_total,
            n_eligible,
            n_storm,
            n_hail_storm,
            storm_hail_grid[
                "period_id"
            ]
            .nunique(),
        ],
    }
)


display(
    interface_audit
)


assert n_total == 504
assert n_eligible == 169
assert n_storm == 123
assert n_hail_storm == 13

assert (
    len(
        storm_conditioned_sample
    )
    == n_storm
)

,quantity,value
0,Total grid-hours,504
1,Coverage-eligible grid-hours,169
2,Strict storm grid-hours,123
3,Observed hail storms,13
4,Primary periods,3


## 3. Construct the two retrospective modeling populations

The **direct training population** contains every radar-coverage-eligible grid-hour.

The **storm-conditioned training population** contains only coverage-eligible grid-hours satisfying the frozen strict storm rule.

The outcome is

$$
H \in \{0,1\}.
$$

The storm indicator is

$$
S \in \{0,1\}.
$$

The direct training population therefore contains both:

$$
S=0,H=0
$$

background grid-hours and

$$
S=1,H\in\{0,1\}
$$

storm grid-hours.

The storm-conditioned population contains only

$$
S=1.
$$

Importantly, later model comparisons use the same held-out $S=1$ test rows for both formulations.

In [4]:
# ------------------------------------------------------------
# Direct retrospective population
# ------------------------------------------------------------

direct_pilot = (
    storm_hail_grid[
        storm_hail_grid[
            "coverage_eligible"
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# Storm-conditioned retrospective population
# ------------------------------------------------------------

conditioned_pilot = (
    storm_hail_grid[
        storm_hail_grid[
            "coverage_eligible"
        ]
        &
        storm_hail_grid[
            "storm_indicator"
        ]
    ]
    .copy()
)


for frame in [
    direct_pilot,
    conditioned_pilot,
]:

    frame[
        "H"
    ] = (
        frame[
            "hail_positive"
        ]
        .astype(int)
    )

    frame[
        "S"
    ] = (
        frame[
            "storm_indicator"
        ]
        .astype(int)
    )


sample_audit = pd.DataFrame(
    {
        "sample": [
            "Direct training population",
            "Storm-conditioned population",
        ],

        "rows": [
            len(
                direct_pilot
            ),
            len(
                conditioned_pilot
            ),
        ],

        "hail_positive": [
            int(
                direct_pilot[
                    "H"
                ]
                .sum()
            ),
            int(
                conditioned_pilot[
                    "H"
                ]
                .sum()
            ),
        ],

        "storm_positive": [
            int(
                direct_pilot[
                    "S"
                ]
                .sum()
            ),
            int(
                conditioned_pilot[
                    "S"
                ]
                .sum()
            ),
        ],
    }
)


display(
    sample_audit
)

,sample,rows,hail_positive,storm_positive
0,Direct training population,169,13,123
1,Storm-conditioned population,123,13,123


## 4. ERA5 environmental predictors

The retrospective comparison uses ERA5 reanalysis fields as environmental predictors.

ERA5 files were retrieved separately for each of the three primary validation periods over the same pilot domain and analysis hour used in Notebook 03.

The local input files contain:

**Single-level fields**

- 2 m temperature;
- 2 m dewpoint temperature;
- CAPE;
- convective inhibition (CIN).

**Pressure-level fields**

At 850, 700, 500, and 300 hPa:

- temperature;
- specific humidity;
- zonal wind;
- meridional wind.

The public notebook assumes these local NetCDF files are already available under

```text
data/era5/pilot_2024/

In [5]:
import xarray as xr


ERA5_PILOT_DIR = (
    DATA_DIR
    / "era5"
    / "pilot_2024"
)


PILOT_PERIOD_IDS = (
    storm_hail_grid[
        "period_id"
    ]
    .drop_duplicates()
    .tolist()
)


single_level_files = {
    period_id:
        ERA5_PILOT_DIR
        / f"{period_id}_single.nc"

    for period_id
    in PILOT_PERIOD_IDS
}


pressure_level_files = {
    period_id:
        ERA5_PILOT_DIR
        / f"{period_id}_pressure.nc"

    for period_id
    in PILOT_PERIOD_IDS
}


# ------------------------------------------------------------
# Public-facing input audit
#
# Display filenames only, not machine-specific absolute paths.
# ------------------------------------------------------------

era5_input_rows = []


for period_id in (
    PILOT_PERIOD_IDS
):

    era5_input_rows.append(
        {
            "period_id":
                period_id,

            "single_level_file":
                single_level_files[
                    period_id
                ].name,

            "single_exists":
                single_level_files[
                    period_id
                ].exists(),

            "pressure_level_file":
                pressure_level_files[
                    period_id
                ].name,

            "pressure_exists":
                pressure_level_files[
                    period_id
                ].exists(),
        }
    )


era5_input_audit = pd.DataFrame(
    era5_input_rows
)


display(
    era5_input_audit
)


missing_files = []


for period_id in (
    PILOT_PERIOD_IDS
):

    for path in [
        single_level_files[
            period_id
        ],
        pressure_level_files[
            period_id
        ],
    ]:

        if not path.exists():

            missing_files.append(
                path.name
            )


if missing_files:

    raise FileNotFoundError(
        "Missing local ERA5 inputs: "
        + ", ".join(
            missing_files
        )
    )

,period_id,single_level_file,single_exists,pressure_level_file,pressure_exists
0,active_20240418,active_20240418_single.nc,True,active_20240418_pressure.nc,True
1,active_20240508,active_20240508_single.nc,True,active_20240508_pressure.nc,True
2,active_20240526,active_20240526_single.nc,True,active_20240526_pressure.nc,True


### 4.1 Align ERA5 with the storm grid

For every grid-hour in the frozen Notebook 03 table, the nearest ERA5 grid point is sampled at the corresponding analysis hour.

ERA5 contributes predictor information only.

The MRMS variables used to construct $S$ are not included in this retrospective environmental baseline.

The raw environmental feature set contains 20 ERA5 variables:

- 4 single-level variables;
- 16 pressure-level variables.

Two additional wind-shear magnitudes are derived for the later compact-feature sensitivity analysis:

$$
\text{shear}_{850,500}
=
\sqrt{
(u_{500}-u_{850})^2
+
(v_{500}-v_{850})^2
},
$$

and

$$
\text{shear}_{850,300}
=
\sqrt{
(u_{300}-u_{850})^2
+
(v_{300}-v_{850})^2
}.
$$

In [6]:
def extract_era5_features_for_period(
    period_id,
    grid_df,
):
    """
    Extract ERA5 environmental predictors at the nearest
    ERA5 grid point for every storm-grid cell in one period.
    """

    ds_single = xr.open_dataset(
        single_level_files[
            period_id
        ],
        engine="netcdf4",
    )


    ds_pressure = xr.open_dataset(
        pressure_level_files[
            period_id
        ],
        engine="netcdf4",
    )


    rows = []


    try:

        for _, row in (
            grid_df
            .iterrows()
        ):

            lat = float(
                row[
                    "grid_lat"
                ]
            )

            lon = float(
                row[
                    "grid_lon"
                ]
            )


            single_point = (
                ds_single
                .sel(
                    latitude=lat,
                    longitude=lon,
                    method="nearest",
                )
            )


            pressure_point = (
                ds_pressure
                .sel(
                    latitude=lat,
                    longitude=lon,
                    method="nearest",
                )
            )


            feature_row = {
                "period_id":
                    period_id,

                "grid_lat":
                    row[
                        "grid_lat"
                    ],

                "grid_lon":
                    row[
                        "grid_lon"
                    ],

                "t2m":
                    float(
                        single_point[
                            "t2m"
                        ]
                        .squeeze()
                        .values
                    ),

                "d2m":
                    float(
                        single_point[
                            "d2m"
                        ]
                        .squeeze()
                        .values
                    ),

                "cape":
                    float(
                        single_point[
                            "cape"
                        ]
                        .squeeze()
                        .values
                    ),

                "cin":
                    float(
                        single_point[
                            "cin"
                        ]
                        .squeeze()
                        .values
                    ),
            }


            for level in [
                850,
                700,
                500,
                300,
            ]:

                level_point = (
                    pressure_point
                    .sel(
                        pressure_level=
                            level
                    )
                )


                for variable in [
                    "t",
                    "q",
                    "u",
                    "v",
                ]:

                    feature_row[
                        f"{variable}_{level}"
                    ] = float(
                        level_point[
                            variable
                        ]
                        .squeeze()
                        .values
                    )


            rows.append(
                feature_row
            )


    finally:

        ds_single.close()
        ds_pressure.close()


    return pd.DataFrame(
        rows
    )

In [7]:
era5_feature_tables = []


for period_id in (
    PILOT_PERIOD_IDS
):

    print(
        "Aligning:",
        period_id,
    )


    period_grid = (
        storm_hail_grid[
            storm_hail_grid[
                "period_id"
            ]
            == period_id
        ][
            [
                "period_id",
                "grid_lat",
                "grid_lon",
            ]
        ]
        .copy()
    )


    period_features = (
        extract_era5_features_for_period(
            period_id=
                period_id,

            grid_df=
                period_grid,
        )
    )


    era5_feature_tables.append(
        period_features
    )


era5_features = (
    pd.concat(
        era5_feature_tables,
        ignore_index=True,
    )
)


# ------------------------------------------------------------
# Derived vertical-wind-shear magnitudes
# ------------------------------------------------------------

era5_features[
    "shear_850_500"
] = np.sqrt(
    (
        era5_features[
            "u_500"
        ]
        -
        era5_features[
            "u_850"
        ]
    ) ** 2
    +
    (
        era5_features[
            "v_500"
        ]
        -
        era5_features[
            "v_850"
        ]
    ) ** 2
)


era5_features[
    "shear_850_300"
] = np.sqrt(
    (
        era5_features[
            "u_300"
        ]
        -
        era5_features[
            "u_850"
        ]
    ) ** 2
    +
    (
        era5_features[
            "v_300"
        ]
        -
        era5_features[
            "v_850"
        ]
    ) ** 2
)


duplicate_era5_rows = int(
    era5_features
    .duplicated(
        subset=[
            "period_id",
            "grid_lat",
            "grid_lon",
        ]
    )
    .sum()
)


print(
    "ERA5 feature rows:",
    len(
        era5_features
    ),
)

print(
    "Duplicate period-grid rows:",
    duplicate_era5_rows,
)


assert (
    len(
        era5_features
    )
    ==
    len(
        storm_hail_grid
    )
)

assert duplicate_era5_rows == 0

Aligning: active_20240418
Aligning: active_20240508
Aligning: active_20240526
ERA5 feature rows: 504
Duplicate period-grid rows: 0


### 4.2 Primary and compact feature sets

Two ERA5 predictor sets are retained for different purposes.

#### Primary 19-feature set

The first baseline preserves the broad environmental specification used in the initial pilot.

The downloaded ERA5 table contains 20 raw environmental variables, but CIN contains structurally missing values in part of the pilot domain.

CIN is therefore excluded rather than:

- treating missing CIN as zero;
- imputing an arbitrary value;
- dropping otherwise valid grid-hours.

The remaining 19 complete ERA5 variables define the **primary feature set**.

#### Compact 6-feature set

A smaller physics-based sensitivity set is also defined:

- CAPE;
- 2 m temperature;
- 2 m dewpoint;
- 500-hPa temperature;
- 850–500-hPa wind shear;
- 850–300-hPa wind shear.

This compact specification reduces feature dimensionality relative to the very small number of hail-positive observations.

It is used as a robustness check rather than selected according to predictive performance.

In [8]:
RAW_ERA5_FEATURE_COLUMNS = [
    "t2m",
    "d2m",
    "cape",
    "cin",

    "t_850",
    "q_850",
    "u_850",
    "v_850",

    "t_700",
    "q_700",
    "u_700",
    "v_700",

    "t_500",
    "q_500",
    "u_500",
    "v_500",

    "t_300",
    "q_300",
    "u_300",
    "v_300",
]


PRIMARY_FEATURE_COLUMNS = [
    feature
    for feature
    in RAW_ERA5_FEATURE_COLUMNS
    if feature != "cin"
]


COMPACT_FEATURE_COLUMNS = [
    "cape",
    "t2m",
    "d2m",
    "t_500",
    "shear_850_500",
    "shear_850_300",
]


# ------------------------------------------------------------
# Missingness audit
# ------------------------------------------------------------

feature_audit_columns = (
    RAW_ERA5_FEATURE_COLUMNS
    + [
        "shear_850_500",
        "shear_850_300",
    ]
)


missing_by_feature = (
    era5_features[
        feature_audit_columns
    ]
    .isna()
    .sum()
)


nonzero_missing = (
    missing_by_feature[
        missing_by_feature
        > 0
    ]
)


print(
    "Primary feature count:",
    len(
        PRIMARY_FEATURE_COLUMNS
    ),
)

print(
    "Compact feature count:",
    len(
        COMPACT_FEATURE_COLUMNS
    ),
)

print(
    "\nFeatures with missing values:"
)


display(
    nonzero_missing
)


primary_missing = int(
    era5_features[
        PRIMARY_FEATURE_COLUMNS
    ]
    .isna()
    .sum()
    .sum()
)


compact_missing = int(
    era5_features[
        COMPACT_FEATURE_COLUMNS
    ]
    .isna()
    .sum()
    .sum()
)


print(
    "Missing values in primary set:",
    primary_missing,
)

print(
    "Missing values in compact set:",
    compact_missing,
)


# ------------------------------------------------------------
# The historical pilot had missingness only in CIN.
# Any additional missing predictor requires investigation.
# ------------------------------------------------------------

unexpected_missing = [
    feature
    for feature
    in nonzero_missing.index
    if feature != "cin"
]


if unexpected_missing:

    raise ValueError(
        "Unexpected ERA5 missingness in: "
        + ", ".join(
            unexpected_missing
        )
    )


assert len(
    PRIMARY_FEATURE_COLUMNS
) == 19

assert len(
    COMPACT_FEATURE_COLUMNS
) == 6

assert primary_missing == 0
assert compact_missing == 0

Primary feature count: 19
Compact feature count: 6

Features with missing values:


cin    85
dtype: int64

Missing values in primary set: 0
Missing values in compact set: 0


In [9]:
# ------------------------------------------------------------
# Merge ERA5 predictors onto the frozen Notebook 03 grid
# ------------------------------------------------------------

pilot_modeling_grid = (
    storm_hail_grid
    .merge(
        era5_features,
        on=[
            "period_id",
            "grid_lat",
            "grid_lon",
        ],
        how="left",
        validate="one_to_one",
    )
)


pilot_modeling_grid[
    "H"
] = (
    pilot_modeling_grid[
        "hail_positive"
    ]
    .astype(int)
)


pilot_modeling_grid[
    "S"
] = (
    pilot_modeling_grid[
        "storm_indicator"
    ]
    .astype(int)
)


direct_model_df = (
    pilot_modeling_grid[
        pilot_modeling_grid[
            "coverage_eligible"
        ]
    ]
    .copy()
)


conditioned_model_df = (
    pilot_modeling_grid[
        pilot_modeling_grid[
            "coverage_eligible"
        ]
        &
        pilot_modeling_grid[
            "storm_indicator"
        ]
    ]
    .copy()
)


modeling_audit = pd.DataFrame(
    {
        "sample": [
            "Direct",
            "Storm-conditioned",
        ],

        "rows": [
            len(
                direct_model_df
            ),
            len(
                conditioned_model_df
            ),
        ],

        "hail_positive": [
            int(
                direct_model_df[
                    "H"
                ]
                .sum()
            ),
            int(
                conditioned_model_df[
                    "H"
                ]
                .sum()
            ),
        ],

        "hail_fraction": [
            float(
                direct_model_df[
                    "H"
                ]
                .mean()
            ),
            float(
                conditioned_model_df[
                    "H"
                ]
                .mean()
            ),
        ],
    }
)


display(
    modeling_audit
)


assert len(
    pilot_modeling_grid
) == 504

assert len(
    direct_model_df
) == 169

assert len(
    conditioned_model_df
) == 123

assert int(
    direct_model_df[
        "H"
    ].sum()
) == 13

assert int(
    conditioned_model_df[
        "H"
    ].sum()
) == 13

,sample,rows,hail_positive,hail_fraction
0,Direct,169,13,0.076923
1,Storm-conditioned,123,13,0.105691


## 5. Leave-one-period-out logistic comparison

The first modeling experiment compares direct and storm-conditioned logistic regression under leave-one-period-out validation.

For each fold:

1. two hail-active periods are used for training;
2. the remaining period is held out completely;
3. the direct model is trained on all coverage-eligible training grid-hours;
4. the storm-conditioned model is trained only on $S=1$ training grid-hours;
5. both models are evaluated on the **same held-out $S=1$ grid-hours**.

The common test population is essential.

Because the two formulations use different training populations, evaluating them on different test populations would confound model performance with differences in event prevalence.

The first comparison uses the 19 complete ERA5 predictors.

A lower-dimensional six-feature specification is then used as a robustness check.

In [10]:
def make_logistic_model(
    C,
):
    """
    Standardized L2-regularized logistic regression.
    """

    return Pipeline(
        [
            (
                "scaler",
                StandardScaler(),
            ),

            (
                "model",
                LogisticRegression(
                    penalty="l2",
                    C=C,
                    max_iter=5000,
                    solver="liblinear",
                ),
            ),
        ]
    )


def binary_metrics(
    y_true,
    probability,
):
    """
    Binary probabilistic classification metrics.
    """

    return {
        "roc_auc":
            roc_auc_score(
                y_true,
                probability,
            ),

        "pr_auc":
            average_precision_score(
                y_true,
                probability,
            ),

        "brier":
            brier_score_loss(
                y_true,
                probability,
            ),

        "log_loss":
            log_loss(
                y_true,
                probability,
                labels=[
                    0,
                    1,
                ],
            ),
    }


def run_direct_conditioned_lopo(
    feature_columns,
    model_c,
):
    """
    Compare direct and storm-conditioned logistic models
    using leave-one-period-out validation.

    Both models are evaluated on the same held-out S=1
    grid-hours.
    """

    fold_rows = []
    prediction_tables = []


    for test_period in (
        PILOT_PERIOD_IDS
    ):

        train_periods = [
            period_id
            for period_id
            in PILOT_PERIOD_IDS
            if period_id
            != test_period
        ]


        # ----------------------------------------------------
        # Training populations
        # ----------------------------------------------------

        direct_train = (
            direct_model_df[
                direct_model_df[
                    "period_id"
                ]
                .isin(
                    train_periods
                )
            ]
            .copy()
        )


        conditioned_train = (
            conditioned_model_df[
                conditioned_model_df[
                    "period_id"
                ]
                .isin(
                    train_periods
                )
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Common held-out S=1 evaluation population
        # ----------------------------------------------------

        common_test = (
            conditioned_model_df[
                conditioned_model_df[
                    "period_id"
                ]
                == test_period
            ]
            .copy()
        )


        X_direct_train = (
            direct_train[
                feature_columns
            ]
        )

        y_direct_train = (
            direct_train[
                "H"
            ]
        )


        X_conditioned_train = (
            conditioned_train[
                feature_columns
            ]
        )

        y_conditioned_train = (
            conditioned_train[
                "H"
            ]
        )


        X_test = (
            common_test[
                feature_columns
            ]
        )

        y_test = (
            common_test[
                "H"
            ]
        )


        # ----------------------------------------------------
        # Fit
        # ----------------------------------------------------

        direct_model = (
            make_logistic_model(
                C=model_c
            )
        )


        conditioned_model = (
            make_logistic_model(
                C=model_c
            )
        )


        direct_model.fit(
            X_direct_train,
            y_direct_train,
        )


        conditioned_model.fit(
            X_conditioned_train,
            y_conditioned_train,
        )


        # ----------------------------------------------------
        # Predict same test rows
        # ----------------------------------------------------

        p_direct = (
            direct_model
            .predict_proba(
                X_test
            )[:, 1]
        )


        p_conditioned = (
            conditioned_model
            .predict_proba(
                X_test
            )[:, 1]
        )


        # ----------------------------------------------------
        # Fold-level metrics
        # ----------------------------------------------------

        for formulation, probability in [
            (
                "Direct",
                p_direct,
            ),
            (
                "Storm-conditioned",
                p_conditioned,
            ),
        ]:

            metrics = (
                binary_metrics(
                    y_true=
                        y_test,

                    probability=
                        probability,
                )
            )


            fold_rows.append(
                {
                    "test_period":
                        test_period,

                    "formulation":
                        formulation,

                    "n_test":
                        len(
                            y_test
                        ),

                    "n_hail_test":
                        int(
                            y_test.sum()
                        ),

                    **metrics,
                }
            )


        # ----------------------------------------------------
        # Save OOF predictions
        # ----------------------------------------------------

        prediction_table = (
            common_test[
                [
                    "period_id",
                    "grid_lat",
                    "grid_lon",
                    "H",
                ]
            ]
            .copy()
        )


        prediction_table[
            "p_direct"
        ] = p_direct


        prediction_table[
            "p_conditioned"
        ] = p_conditioned


        prediction_tables.append(
            prediction_table
        )


    fold_results = pd.DataFrame(
        fold_rows
    )


    oof_predictions = (
        pd.concat(
            prediction_tables,
            ignore_index=True,
        )
    )


    # --------------------------------------------------------
    # Pooled out-of-period metrics
    # --------------------------------------------------------

    pooled_rows = []


    for formulation, prediction_column in [
        (
            "Direct",
            "p_direct",
        ),
        (
            "Storm-conditioned",
            "p_conditioned",
        ),
    ]:

        metrics = binary_metrics(
            y_true=
                oof_predictions[
                    "H"
                ],

            probability=
                oof_predictions[
                    prediction_column
                ],
        )


        pooled_rows.append(
            {
                "formulation":
                    formulation,

                "n":
                    len(
                        oof_predictions
                    ),

                "hail_positive":
                    int(
                        oof_predictions[
                            "H"
                        ]
                        .sum()
                    ),

                **metrics,
            }
        )


    pooled_results = pd.DataFrame(
        pooled_rows
    )


    return (
        fold_results,
        oof_predictions,
        pooled_results,
    )

In [11]:
(
    primary_fold_results,
    primary_oof_predictions,
    primary_pooled_results,
) = run_direct_conditioned_lopo(
    feature_columns=
        PRIMARY_FEATURE_COLUMNS,

    model_c=
        1.0,
)


print(
    "19-FEATURE ERA5 — PERIOD LEVEL"
)

display(
    primary_fold_results
)


print(
    "\n19-FEATURE ERA5 — POOLED OOF"
)

display(
    primary_pooled_results
)

19-FEATURE ERA5 — PERIOD LEVEL


,test_period,formulation,n_test,n_hail_test,roc_auc,pr_auc,brier,log_loss
0,active_20240418,Direct,45,2,0.627907,0.107143,0.299091,0.823663
1,active_20240418,Storm-conditioned,45,2,0.744186,0.212121,0.143962,0.436370
2,active_20240508,Direct,40,4,0.597222,0.166088,0.092477,0.345702
3,active_20240508,Storm-conditioned,40,4,0.451389,0.130247,0.092200,0.335355
4,active_20240526,Direct,38,7,0.525346,0.208489,0.241398,0.677061
5,active_20240526,Storm-conditioned,38,7,0.552995,0.222942,0.266970,0.730299



19-FEATURE ERA5 — POOLED OOF


,formulation,n,hail_positive,roc_auc,pr_auc,brier,log_loss
0,Direct,123,13,0.534266,0.121632,0.214076,0.622937
1,Storm-conditioned,123,13,0.627273,0.177797,0.165131,0.494327


### 5.1 Initial 19-feature result

Under the broad 19-variable ERA5 specification, the storm-conditioned logistic model performs better than the direct model on all four pooled pilot metrics.

Using the finalized storm-first sample from Notebook 03, the pooled out-of-period results are approximately:

- ROC-AUC: 0.534 → 0.627;
- PR-AUC: 0.122 → 0.178;
- Brier score: 0.214 → 0.165;
- log loss: 0.623 → 0.494.

Because both formulations are evaluated on exactly the same held-out storm grid-hours, this difference is not caused by different test-set prevalence.

However, the result is not yet strong evidence that storm conditioning itself improves hail prediction.

The model uses 19 environmental predictors but contains only 13 hail-positive storm observations across three hail-selected periods.

A lower-dimensional specification is therefore used to test whether the apparent advantage is robust to feature-set complexity and regularization.

## 6. Compact-feature robustness check

The direct-versus-conditioned comparison is repeated using the six-feature physics-based predictor set defined earlier:

- CAPE;
- 2 m temperature;
- 2 m dewpoint temperature;
- 500-hPa temperature;
- 850–500-hPa wind shear;
- 850–300-hPa wind shear.

This specification was defined to reduce dimensionality rather than to optimize predictive performance.

Stronger L2 regularization is also used:

$$
C = 0.1.
$$

The validation design remains unchanged:

- leave one full period out;
- train direct and storm-conditioned models separately;
- evaluate both models on the same held-out $S=1$ population.

In [12]:
(
    compact_fold_results,
    compact_oof_predictions,
    compact_pooled_results,
) = run_direct_conditioned_lopo(
    feature_columns=
        COMPACT_FEATURE_COLUMNS,

    model_c=
        0.1,
)


print(
    "COMPACT ERA5 — PERIOD LEVEL"
)

display(
    compact_fold_results
)


print(
    "\nCOMPACT ERA5 — POOLED OOF"
)

display(
    compact_pooled_results
)

COMPACT ERA5 — PERIOD LEVEL


,test_period,formulation,n_test,n_hail_test,roc_auc,pr_auc,brier,log_loss
0,active_20240418,Direct,45,2,0.453488,0.058574,0.121403,0.421397
1,active_20240418,Storm-conditioned,45,2,0.593023,0.085714,0.161465,0.508244
2,active_20240508,Direct,40,4,0.895833,0.591667,0.087221,0.314262
3,active_20240508,Storm-conditioned,40,4,0.770833,0.312500,0.095727,0.345927
4,active_20240526,Direct,38,7,0.373272,0.159457,0.156248,0.494281
5,active_20240526,Storm-conditioned,38,7,0.327189,0.149873,0.161510,0.507434



COMPACT ERA5 — POOLED OOF


,formulation,n,hail_positive,roc_auc,pr_auc,brier,log_loss
0,Direct,123,13,0.474825,0.099039,0.121052,0.409073
1,Storm-conditioned,123,13,0.452448,0.098656,0.140101,0.455208


In [13]:
comparison_rows = []


for feature_set, results in [
    (
        "19-feature ERA5",
        primary_pooled_results,
    ),
    (
        "Compact 6-feature ERA5",
        compact_pooled_results,
    ),
]:

    indexed = (
        results
        .set_index(
            "formulation"
        )
    )


    direct = (
        indexed
        .loc[
            "Direct"
        ]
    )


    conditioned = (
        indexed
        .loc[
            "Storm-conditioned"
        ]
    )


    comparison_rows.append(
        {
            "feature_set":
                feature_set,

            "direct_roc_auc":
                direct[
                    "roc_auc"
                ],

            "conditioned_roc_auc":
                conditioned[
                    "roc_auc"
                ],

            "delta_roc_auc":
                (
                    conditioned[
                        "roc_auc"
                    ]
                    -
                    direct[
                        "roc_auc"
                    ]
                ),

            "direct_pr_auc":
                direct[
                    "pr_auc"
                ],

            "conditioned_pr_auc":
                conditioned[
                    "pr_auc"
                ],

            "delta_pr_auc":
                (
                    conditioned[
                        "pr_auc"
                    ]
                    -
                    direct[
                        "pr_auc"
                    ]
                ),

            "direct_brier":
                direct[
                    "brier"
                ],

            "conditioned_brier":
                conditioned[
                    "brier"
                ],

            "delta_brier":
                (
                    direct[
                        "brier"
                    ]
                    -
                    conditioned[
                        "brier"
                    ]
                ),

            "direct_log_loss":
                direct[
                    "log_loss"
                ],

            "conditioned_log_loss":
                conditioned[
                    "log_loss"
                ],

            "delta_log_loss":
                (
                    direct[
                        "log_loss"
                    ]
                    -
                    conditioned[
                        "log_loss"
                    ]
                ),
        }
    )


robustness_summary = pd.DataFrame(
    comparison_rows
)


print(
    "Positive delta = storm conditioning better"
)

display(
    robustness_summary
)

Positive delta = storm conditioning better


,feature_set,direct_roc_auc,conditioned_roc_auc,delta_roc_auc,direct_pr_auc,conditioned_pr_auc,delta_pr_auc,direct_brier,conditioned_brier,delta_brier,direct_log_loss,conditioned_log_loss,delta_log_loss
0,19-feature ERA5,0.534266,0.627273,0.093007,0.121632,0.177797,0.056165,0.214076,0.165131,0.048944,0.622937,0.494327,0.128610
1,Compact 6-feature ERA5,0.474825,0.452448,-0.022378,0.099039,0.098656,-0.000382,0.121052,0.140101,-0.019049,0.409073,0.455208,-0.046135


### 6.1 Robustness interpretation

The apparent advantage of storm conditioning is not stable across reasonable ERA5 model specifications.

The broad 19-feature logistic model favors storm conditioning on all four pooled metrics.

In contrast, under the compact six-feature specification:

- ROC-AUC is lower for the storm-conditioned model;
- PR-AUC is nearly unchanged;
- Brier score is worse;
- log loss is worse.

The initial conditioning advantage therefore cannot be interpreted as a robust consequence of restricting the training population to radar-defined storms.

One possible explanation is that the background grid-hours available to the direct formulation contain useful information about the environmental gradient from quiet conditions to convection and hail.

A second possibility is that coarse ERA5 environmental predictors contain limited residual information for distinguishing hail-producing storms from non-hail storms once storm occurrence has already been established.

The next analysis tests this interpretation more directly by changing the composition of the negative training sample while holding the positive cases and storm-only evaluation population fixed.

## 7. Negative-composition falsification

The compact-feature result suggests that storm conditioning may not provide a robust predictive advantage when only coarse ERA5 environmental predictors are available.

A falsification experiment is used to test that interpretation more directly.

The experiment holds fixed:

- the hail-positive training observations;
- the number of negative training observations;
- the six compact ERA5 predictors;
- the logistic-regression specification;
- the leave-one-period-out split;
- the held-out $S=1$ evaluation population.

Only the **composition of the negative training sample** is changed.

### True storm-negative model

The reference storm-conditioned model uses:

$$
S=1,H=1
$$

as positives and

$$
S=1,H=0
$$

as negatives.

### Random-negative placebo

For each training fold, the same hail-positive storm observations are retained.

The storm negatives are then replaced by the same number of randomly sampled

$$
H=0
$$

grid-hours from the broader coverage-eligible training population.

This placebo pool can contain both radar-defined storms and background grid-hours.

The procedure is repeated 300 times.

If true storm negatives contain uniquely useful ERA5 information for within-storm hail discrimination, the true storm-negative model should consistently outperform the random-negative placebo distribution on the common storm-only test population.

In [14]:
# ------------------------------------------------------------
# Common dataframe for negative-composition experiments
# ------------------------------------------------------------

falsification_df = (
    pilot_modeling_grid[
        pilot_modeling_grid[
            "coverage_eligible"
        ]
    ]
    .copy()
)


falsification_df[
    "H"
] = (
    falsification_df[
        "hail_positive"
    ]
    .astype(int)
)


falsification_df[
    "S"
] = (
    falsification_df[
        "storm_indicator"
    ]
    .astype(int)
)


# ------------------------------------------------------------
# Compact predictors must be complete
# ------------------------------------------------------------

missing_compact = int(
    falsification_df[
        COMPACT_FEATURE_COLUMNS
    ]
    .isna()
    .sum()
    .sum()
)


print(
    "Falsification rows:",
    len(
        falsification_df
    ),
)

print(
    "Storm-positive rows:",
    int(
        falsification_df[
            "S"
        ]
        .sum()
    ),
)

print(
    "Hail-positive rows:",
    int(
        falsification_df[
            "H"
        ]
        .sum()
    ),
)

print(
    "Missing compact predictors:",
    missing_compact,
)


assert len(
    falsification_df
) == 169

assert int(
    falsification_df[
        "S"
    ].sum()
) == 123

assert int(
    falsification_df[
        "H"
    ].sum()
) == 13

assert missing_compact == 0


FALSIFICATION_REPS = 300
RANDOM_SEED = 42

Falsification rows: 169
Storm-positive rows: 123
Hail-positive rows: 13
Missing compact predictors: 0


In [15]:
# ------------------------------------------------------------
# Reference: true storm-negative compact model
#
# These are the already-computed LOPO predictions from
# Section 6.
# ------------------------------------------------------------

true_storm_metrics = (
    binary_metrics(
        y_true=
            compact_oof_predictions[
                "H"
            ],

        probability=
            compact_oof_predictions[
                "p_conditioned"
            ],
    )
)


print(
    "TRUE STORM-NEGATIVE MODEL"
)

display(
    pd.Series(
        true_storm_metrics
    )
)


# ------------------------------------------------------------
# Random-negative placebo
#
# Same storm-hail positives.
# Same number of negatives as the true storm-conditioned
# training sample.
# Same held-out S=1 test rows.
# Only negative composition changes.
# ------------------------------------------------------------

rng = np.random.default_rng(
    RANDOM_SEED
)


placebo_rows = []


for rep in range(
    FALSIFICATION_REPS
):

    rep_predictions = []


    for test_period in (
        PILOT_PERIOD_IDS
    ):

        train = (
            falsification_df[
                falsification_df[
                    "period_id"
                ]
                != test_period
            ]
            .copy()
        )


        test = (
            falsification_df[
                (
                    falsification_df[
                        "period_id"
                    ]
                    == test_period
                )
                &
                (
                    falsification_df[
                        "S"
                    ]
                    == 1
                )
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Fixed positive cases
        # ----------------------------------------------------

        train_positive = (
            train[
                (
                    train[
                        "S"
                    ]
                    == 1
                )
                &
                (
                    train[
                        "H"
                    ]
                    == 1
                )
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Match the number of negatives in the actual
        # storm-conditioned training population
        # ----------------------------------------------------

        n_storm_negative = int(
            (
                (
                    train[
                        "S"
                    ]
                    == 1
                )
                &
                (
                    train[
                        "H"
                    ]
                    == 0
                )
            )
            .sum()
        )


        random_negative_pool = (
            train[
                train[
                    "H"
                ]
                == 0
            ]
        )


        if (
            len(
                random_negative_pool
            )
            < n_storm_negative
        ):

            raise ValueError(
                "Random-negative pool is smaller "
                "than required negative sample."
            )


        random_negatives = (
            random_negative_pool
            .sample(
                n=
                    n_storm_negative,

                replace=False,

                random_state=int(
                    rng.integers(
                        0,
                        2**31 - 1,
                    )
                ),
            )
        )


        placebo_train = (
            pd.concat(
                [
                    train_positive,
                    random_negatives,
                ],
                ignore_index=True,
            )
        )


        model = make_logistic_model(
            C=0.1
        )


        model.fit(
            placebo_train[
                COMPACT_FEATURE_COLUMNS
            ],
            placebo_train[
                "H"
            ],
        )


        probability = (
            model
            .predict_proba(
                test[
                    COMPACT_FEATURE_COLUMNS
                ]
            )[:, 1]
        )


        prediction_table = (
            test[
                [
                    "period_id",
                    "H",
                ]
            ]
            .copy()
        )


        prediction_table[
            "probability"
        ] = probability


        rep_predictions.append(
            prediction_table
        )


    rep_predictions = (
        pd.concat(
            rep_predictions,
            ignore_index=True,
        )
    )


    rep_metrics = (
        binary_metrics(
            y_true=
                rep_predictions[
                    "H"
                ],

            probability=
                rep_predictions[
                    "probability"
                ],
        )
    )


    placebo_rows.append(
        {
            "rep":
                rep,

            **rep_metrics,
        }
    )


random_negative_placebo = (
    pd.DataFrame(
        placebo_rows
    )
)


# ------------------------------------------------------------
# Summarize placebo distribution
# ------------------------------------------------------------

placebo_summary = pd.DataFrame(
    {
        "true_storm_negative": [
            true_storm_metrics[
                "roc_auc"
            ],
            true_storm_metrics[
                "pr_auc"
            ],
            true_storm_metrics[
                "brier"
            ],
            true_storm_metrics[
                "log_loss"
            ],
        ],

        "placebo_median": [
            random_negative_placebo[
                "roc_auc"
            ].median(),

            random_negative_placebo[
                "pr_auc"
            ].median(),

            random_negative_placebo[
                "brier"
            ].median(),

            random_negative_placebo[
                "log_loss"
            ].median(),
        ],

        "placebo_q05": [
            random_negative_placebo[
                "roc_auc"
            ].quantile(
                0.05
            ),

            random_negative_placebo[
                "pr_auc"
            ].quantile(
                0.05
            ),

            random_negative_placebo[
                "brier"
            ].quantile(
                0.05
            ),

            random_negative_placebo[
                "log_loss"
            ].quantile(
                0.05
            ),
        ],

        "placebo_q95": [
            random_negative_placebo[
                "roc_auc"
            ].quantile(
                0.95
            ),

            random_negative_placebo[
                "pr_auc"
            ].quantile(
                0.95
            ),

            random_negative_placebo[
                "brier"
            ].quantile(
                0.95
            ),

            random_negative_placebo[
                "log_loss"
            ].quantile(
                0.95
            ),
        ],
    },

    index=[
        "ROC-AUC",
        "PR-AUC",
        "Brier",
        "Log loss",
    ],
)


display(
    placebo_summary
)


print(
    "\nFraction of placebo runs beaten "
    "by TRUE storm negatives:"
)


print(
    "ROC-AUC:",
    round(
        (
            random_negative_placebo[
                "roc_auc"
            ]
            <
            true_storm_metrics[
                "roc_auc"
            ]
        )
        .mean(),
        3,
    ),
)


print(
    "PR-AUC:",
    round(
        (
            random_negative_placebo[
                "pr_auc"
            ]
            <
            true_storm_metrics[
                "pr_auc"
            ]
        )
        .mean(),
        3,
    ),
)


print(
    "Brier:",
    round(
        (
            random_negative_placebo[
                "brier"
            ]
            >
            true_storm_metrics[
                "brier"
            ]
        )
        .mean(),
        3,
    ),
)


print(
    "Log loss:",
    round(
        (
            random_negative_placebo[
                "log_loss"
            ]
            >
            true_storm_metrics[
                "log_loss"
            ]
        )
        .mean(),
        3,
    ),
)

TRUE STORM-NEGATIVE MODEL


roc_auc     0.452448
pr_auc      0.098656
brier       0.140101
log_loss    0.455208
dtype: float64

,true_storm_negative,placebo_median,placebo_q05,placebo_q95
ROC-AUC,0.452448,0.480420,0.452413,0.508427
PR-AUC,0.098656,0.101394,0.095449,0.107795
Brier,0.140101,0.140453,0.131392,0.154690
Log loss,0.455208,0.453920,0.434809,0.483291



Fraction of placebo runs beaten by TRUE storm negatives:
ROC-AUC: 0.05
PR-AUC: 0.253
Brier: 0.52
Log loss: 0.467


### 7.1 Random-negative placebo result

Using the finalized Notebook 03 sample, the true storm-negative model does not outperform the random-negative placebo distribution.

The true storm-negative compact model yields approximately:

- ROC-AUC: 0.452;
- PR-AUC: 0.099;
- Brier score: 0.140;
- log loss: 0.455.

Across 300 random-negative placebo repetitions, the median performance is approximately:

- ROC-AUC: 0.480;
- PR-AUC: 0.101;
- Brier score: 0.140;
- log loss: 0.454.

The true storm-negative model beats only about 5% of placebo runs in ROC-AUC and about 25% in PR-AUC.

For Brier score and log loss, its performance is close to the center of the placebo distribution.

This result provides no evidence that restricting negative training examples to radar-defined storms gives compact ERA5 predictors a uniquely informative within-storm hail signal.

A stricter follow-up comparison therefore contrasts **pure storm negatives** against **pure background negatives** while holding positive cases, negative sample size, model form, test population, and probability calibration rules fixed.

### 7.2 Pure storm negatives versus pure background negatives

The random-negative placebo mixes storm and background negative cases.

A more targeted falsification compares two deliberately different negative populations.

For every leave-one-period-out fold, both models use the same hail-positive storm observations:

$$
S=1,H=1.
$$

The first model is trained with sampled **storm negatives**:

$$
S=1,H=0.
$$

The second model is trained with sampled **background negatives**:

$$
S=0,H=0.
$$

The number of negatives is forced to be identical between the two models within each fold.

Because this equal-size sampling changes the class prevalence relative to the natural storm training population, both models receive the same form of prior-probability correction.

For each fitted model, the sample prevalence is

$$
\pi_{\text{sample}},
$$

while the target prevalence is the observed hail fraction among all radar-defined training storms,

$$
\pi_{\text{target}}
=
P(H=1 \mid S=1)
$$

within the training periods.

Predicted log odds are adjusted by

$$
\operatorname{logit}(p_{\text{corrected}})
=
\operatorname{logit}(p_{\text{sample}})
+
\operatorname{logit}(\pi_{\text{target}})
-
\operatorname{logit}(\pi_{\text{sample}}).
$$

This correction addresses the artificial **class-prior shift** induced by negative subsampling.

It does not remove the covariate-distribution difference between storm and background negatives. The corrected probabilities are therefore used as a diagnostic comparison rather than interpreted as fully population-calibrated probabilities.

The experiment is repeated 300 times.

Both models are always evaluated on the same held-out $S=1$ population.

In [16]:
# ------------------------------------------------------------
# Pure storm negatives versus pure background negatives
#
# Same positives
# Same negative sample size
# Same compact model
# Same held-out S=1 test population
# Symmetric prevalence correction
# ------------------------------------------------------------

def prior_correct_probability(
    probability,
    sample_prevalence,
    target_prevalence,
    eps=1e-6,
):
    """
    Correct predicted probabilities for a difference between
    artificial training prevalence and target prevalence.
    """

    probability = np.clip(
        probability,
        eps,
        1 - eps,
    )

    sample_prevalence = np.clip(
        sample_prevalence,
        eps,
        1 - eps,
    )

    target_prevalence = np.clip(
        target_prevalence,
        eps,
        1 - eps,
    )


    sample_logit = np.log(
        sample_prevalence
        /
        (
            1
            -
            sample_prevalence
        )
    )


    target_logit = np.log(
        target_prevalence
        /
        (
            1
            -
            target_prevalence
        )
    )


    probability_logit = np.log(
        probability
        /
        (
            1
            -
            probability
        )
    )


    corrected_logit = (
        probability_logit
        +
        target_logit
        -
        sample_logit
    )


    return (
        1
        /
        (
            1
            +
            np.exp(
                -corrected_logit
            )
        )
    )


rng = np.random.default_rng(
    RANDOM_SEED
)


pure_negative_rows = []


for rep in range(
    FALSIFICATION_REPS
):

    storm_predictions = []
    background_predictions = []


    for test_period in (
        PILOT_PERIOD_IDS
    ):

        train = (
            falsification_df[
                falsification_df[
                    "period_id"
                ]
                != test_period
            ]
            .copy()
        )


        test = (
            falsification_df[
                (
                    falsification_df[
                        "period_id"
                    ]
                    == test_period
                )
                &
                (
                    falsification_df[
                        "S"
                    ]
                    == 1
                )
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Fixed hail-positive storm cases
        # ----------------------------------------------------

        train_positive = (
            train[
                (
                    train[
                        "S"
                    ]
                    == 1
                )
                &
                (
                    train[
                        "H"
                    ]
                    == 1
                )
            ]
            .copy()
        )


        storm_negative_pool = (
            train[
                (
                    train[
                        "S"
                    ]
                    == 1
                )
                &
                (
                    train[
                        "H"
                    ]
                    == 0
                )
            ]
            .copy()
        )


        background_negative_pool = (
            train[
                (
                    train[
                        "S"
                    ]
                    == 0
                )
                &
                (
                    train[
                        "H"
                    ]
                    == 0
                )
            ]
            .copy()
        )


        # ----------------------------------------------------
        # Equal negative sample size between arms
        # ----------------------------------------------------

        n_negative = min(
            len(
                storm_negative_pool
            ),
            len(
                background_negative_pool
            ),
        )


        if n_negative == 0:

            raise ValueError(
                "No negative observations available "
                f"for fold {test_period}."
            )


        storm_negatives = (
            storm_negative_pool
            .sample(
                n=
                    n_negative,

                replace=False,

                random_state=int(
                    rng.integers(
                        0,
                        2**31 - 1,
                    )
                ),
            )
        )


        background_negatives = (
            background_negative_pool
            .sample(
                n=
                    n_negative,

                replace=False,

                random_state=int(
                    rng.integers(
                        0,
                        2**31 - 1,
                    )
                ),
            )
        )


        storm_train = (
            pd.concat(
                [
                    train_positive,
                    storm_negatives,
                ],
                ignore_index=True,
            )
        )


        background_train = (
            pd.concat(
                [
                    train_positive,
                    background_negatives,
                ],
                ignore_index=True,
            )
        )


        # ----------------------------------------------------
        # Same artificial sample prevalence in both arms
        # ----------------------------------------------------

        storm_sample_prevalence = float(
            storm_train[
                "H"
            ]
            .mean()
        )


        background_sample_prevalence = float(
            background_train[
                "H"
            ]
            .mean()
        )


        # ----------------------------------------------------
        # Target prevalence:
        # observed hail fraction among all training storms
        # ----------------------------------------------------

        training_storm_population = (
            train[
                train[
                    "S"
                ]
                == 1
            ]
        )


        target_prevalence = float(
            training_storm_population[
                "H"
            ]
            .mean()
        )


        # ----------------------------------------------------
        # Fit storm-negative arm
        # ----------------------------------------------------

        storm_model = make_logistic_model(
            C=0.1
        )


        storm_model.fit(
            storm_train[
                COMPACT_FEATURE_COLUMNS
            ],
            storm_train[
                "H"
            ],
        )


        p_storm_raw = (
            storm_model
            .predict_proba(
                test[
                    COMPACT_FEATURE_COLUMNS
                ]
            )[:, 1]
        )


        p_storm = prior_correct_probability(
            probability=
                p_storm_raw,

            sample_prevalence=
                storm_sample_prevalence,

            target_prevalence=
                target_prevalence,
        )


        # ----------------------------------------------------
        # Fit background-negative arm
        # ----------------------------------------------------

        background_model = (
            make_logistic_model(
                C=0.1
            )
        )


        background_model.fit(
            background_train[
                COMPACT_FEATURE_COLUMNS
            ],
            background_train[
                "H"
            ],
        )


        p_background_raw = (
            background_model
            .predict_proba(
                test[
                    COMPACT_FEATURE_COLUMNS
                ]
            )[:, 1]
        )


        p_background = (
            prior_correct_probability(
                probability=
                    p_background_raw,

                sample_prevalence=
                    background_sample_prevalence,

                target_prevalence=
                    target_prevalence,
            )
        )


        # ----------------------------------------------------
        # Save common held-out predictions
        # ----------------------------------------------------

        storm_out = (
            test[
                [
                    "period_id",
                    "H",
                ]
            ]
            .copy()
        )


        storm_out[
            "probability"
        ] = p_storm


        background_out = (
            test[
                [
                    "period_id",
                    "H",
                ]
            ]
            .copy()
        )


        background_out[
            "probability"
        ] = p_background


        storm_predictions.append(
            storm_out
        )


        background_predictions.append(
            background_out
        )


    # --------------------------------------------------------
    # Pool OOF predictions across periods
    # --------------------------------------------------------

    storm_predictions = (
        pd.concat(
            storm_predictions,
            ignore_index=True,
        )
    )


    background_predictions = (
        pd.concat(
            background_predictions,
            ignore_index=True,
        )
    )


    storm_metrics = (
        binary_metrics(
            y_true=
                storm_predictions[
                    "H"
                ],

            probability=
                storm_predictions[
                    "probability"
                ],
        )
    )


    background_metrics = (
        binary_metrics(
            y_true=
                background_predictions[
                    "H"
                ],

            probability=
                background_predictions[
                    "probability"
                ],
        )
    )


    pure_negative_rows.append(
        {
            "rep":
                rep,

            "storm_roc_auc":
                storm_metrics[
                    "roc_auc"
                ],

            "background_roc_auc":
                background_metrics[
                    "roc_auc"
                ],

            "storm_pr_auc":
                storm_metrics[
                    "pr_auc"
                ],

            "background_pr_auc":
                background_metrics[
                    "pr_auc"
                ],

            "storm_brier":
                storm_metrics[
                    "brier"
                ],

            "background_brier":
                background_metrics[
                    "brier"
                ],

            "storm_log_loss":
                storm_metrics[
                    "log_loss"
                ],

            "background_log_loss":
                background_metrics[
                    "log_loss"
                ],
        }
    )


pure_negative_comparison = (
    pd.DataFrame(
        pure_negative_rows
    )
)


# ------------------------------------------------------------
# Win rates:
# fraction of repetitions in which storm negatives are better
# ------------------------------------------------------------

storm_negative_win_rates = pd.Series(
    {
        "ROC-AUC":
            (
                pure_negative_comparison[
                    "storm_roc_auc"
                ]
                >
                pure_negative_comparison[
                    "background_roc_auc"
                ]
            )
            .mean(),

        "PR-AUC":
            (
                pure_negative_comparison[
                    "storm_pr_auc"
                ]
                >
                pure_negative_comparison[
                    "background_pr_auc"
                ]
            )
            .mean(),

        "Brier":
            (
                pure_negative_comparison[
                    "storm_brier"
                ]
                <
                pure_negative_comparison[
                    "background_brier"
                ]
            )
            .mean(),

        "Log loss":
            (
                pure_negative_comparison[
                    "storm_log_loss"
                ]
                <
                pure_negative_comparison[
                    "background_log_loss"
                ]
            )
            .mean(),
    },
    name="storm_negative_win_rate",
)


# ------------------------------------------------------------
# Median metrics
# ------------------------------------------------------------

median_negative_metrics = pd.DataFrame(
    {
        "storm_negative": [
            pure_negative_comparison[
                "storm_roc_auc"
            ].median(),

            pure_negative_comparison[
                "storm_pr_auc"
            ].median(),

            pure_negative_comparison[
                "storm_brier"
            ].median(),

            pure_negative_comparison[
                "storm_log_loss"
            ].median(),
        ],

        "background_negative": [
            pure_negative_comparison[
                "background_roc_auc"
            ].median(),

            pure_negative_comparison[
                "background_pr_auc"
            ].median(),

            pure_negative_comparison[
                "background_brier"
            ].median(),

            pure_negative_comparison[
                "background_log_loss"
            ].median(),
        ],
    },

    index=[
        "ROC-AUC",
        "PR-AUC",
        "Brier",
        "Log loss",
    ],
)


print(
    "Fraction of runs in which "
    "STORM negatives beat BACKGROUND negatives:"
)

display(
    storm_negative_win_rates
)


print(
    "\nMedian corrected metrics:"
)

display(
    median_negative_metrics
)

Fraction of runs in which STORM negatives beat BACKGROUND negatives:


ROC-AUC     0.093333
PR-AUC      0.000000
Brier       0.953333
Log loss    0.933333
Name: storm_negative_win_rate, dtype: float64


Median corrected metrics:


,storm_negative,background_negative
ROC-AUC,0.524126,0.628671
PR-AUC,0.111435,0.184722
Brier,0.122753,0.148699
Log loss,0.409246,0.460527


In [17]:
# ------------------------------------------------------------
# Audit negative-pool sizes used by the pure-negative
# falsification
# ------------------------------------------------------------

negative_pool_rows = []


for test_period in (
    PILOT_PERIOD_IDS
):

    train = (
        falsification_df[
            falsification_df[
                "period_id"
            ]
            != test_period
        ]
        .copy()
    )


    n_positive = int(
        (
            (
                train[
                    "S"
                ]
                == 1
            )
            &
            (
                train[
                    "H"
                ]
                == 1
            )
        )
        .sum()
    )


    n_storm_negative = int(
        (
            (
                train[
                    "S"
                ]
                == 1
            )
            &
            (
                train[
                    "H"
                ]
                == 0
            )
        )
        .sum()
    )


    n_background_negative = int(
        (
            (
                train[
                    "S"
                ]
                == 0
            )
            &
            (
                train[
                    "H"
                ]
                == 0
            )
        )
        .sum()
    )


    n_sampled_negative = min(
        n_storm_negative,
        n_background_negative,
    )


    training_storm_population = (
        train[
            train[
                "S"
            ]
            == 1
        ]
    )


    target_prevalence = float(
        training_storm_population[
            "H"
        ]
        .mean()
    )


    sample_prevalence = (
        n_positive
        /
        (
            n_positive
            +
            n_sampled_negative
        )
    )


    negative_pool_rows.append(
        {
            "test_period":
                test_period,

            "hail_positive":
                n_positive,

            "storm_negative_pool":
                n_storm_negative,

            "background_negative_pool":
                n_background_negative,

            "sampled_negatives_per_arm":
                n_sampled_negative,

            "sample_prevalence":
                sample_prevalence,

            "target_storm_prevalence":
                target_prevalence,
        }
    )


negative_pool_audit = pd.DataFrame(
    negative_pool_rows
)


display(
    negative_pool_audit
)

,test_period,hail_positive,storm_negative_pool,background_negative_pool,sampled_negatives_per_arm,sample_prevalence,target_storm_prevalence
0,active_20240418,11,67,33,33,0.250000,0.141026
1,active_20240508,9,74,32,32,0.219512,0.108434
2,active_20240526,6,79,27,27,0.181818,0.070588


### 7.3 Pure-negative falsification result

The pure-negative comparison reveals a tradeoff rather than a single dominant negative population.

After equalizing negative sample size and applying the same prior-probability adjustment to both arms, models trained with **background negatives** show substantially stronger discrimination on the held-out storm population.

Across 300 repetitions:

- storm negatives outperform background negatives in ROC-AUC in only about 9% of runs;
- storm negatives never outperform background negatives in PR-AUC.

Median discrimination is approximately:

$$
\text{ROC-AUC: }
0.524
\text{ versus }
0.629,
$$

and

$$
\text{PR-AUC: }
0.111
\text{ versus }
0.185,
$$

for storm-negative and background-negative training, respectively.

Probability scores show the opposite pattern.

Storm-negative models achieve lower Brier score in about 95% of repetitions and lower log loss in about 93% of repetitions.

Median values are approximately:

$$
\text{Brier: }
0.123
\text{ versus }
0.149,
$$

and

$$
\text{log loss: }
0.409
\text{ versus }
0.461.
$$

The result therefore does **not** establish that either negative population is uniformly superior.

Instead, it suggests that background negatives provide ERA5 with a broader environmental contrast that improves ranking of hail risk, whereas storm negatives produce probabilities that are better matched to the storm-only evaluation population.

Together with the random-negative placebo, this experiment provides no evidence that storm-negative restriction creates a uniquely informative ERA5-only hail classifier.

## 8. Retrospective modeling conclusion

This notebook tested whether a hail-independent radar storm sample frame improves hail discrimination when the predictors are limited to coarse ERA5 environmental fields.

The answer is not a simple yes.

Under the initial 19-feature logistic specification, storm conditioning appears beneficial on all four pooled out-of-period metrics.

That apparent advantage is not robust.

With a compact six-feature specification:

- ROC-AUC decreases under storm conditioning;
- PR-AUC is essentially unchanged;
- Brier score worsens;
- log loss worsens.

The negative-composition falsification provides a direct explanation for this instability.

True storm negatives do not outperform randomly composed negative samples, and pure background negatives produce substantially stronger ROC-AUC and PR-AUC than pure storm negatives.

At the same time, storm-negative models retain better Brier score and log loss on the storm-only evaluation population.

The evidence therefore supports the following interpretation:

> Coarse ERA5 environmental predictors appear to contain substantial information about the environmental transition from background conditions toward convection and hail, but limited residual information for distinguishing hail-producing storms from non-hail storms once storm occurrence has already been established.

This does not invalidate the storm-first sampling framework established in Notebook 03.

Instead, it clarifies **where additional predictive information is required**.

If hail discrimination within storms depends on storm-scale structure rather than only on the large-scale environment, the natural next test is to introduce temporally valid pre-event radar features.

The next notebook therefore changes the problem from this retrospective same-hour diagnostic to an explicit future-window experiment with:

$$
X^-:
\text{ information available by forecast origin }t_0,
$$

$$
S^+:
\text{ future radar-defined storm occurrence},
$$

and

$$
H^+:
\text{ future observed hail occurrence}.
$$

The direct forecast

$$
P(H^+=1\mid X^-)
$$

can then be compared fairly with the hierarchical decomposition

$$
P(S^+=1\mid X^-)
\,
P(H^+=1\mid S^+=1,X^-).
$$

The objective is not to force the hierarchy to outperform the direct model.

The objective is to identify whether predictive information enters primarily through storm occurrence, through hail production within storms, or through both stages.

In [18]:
# ------------------------------------------------------------
# Save compact retrospective modeling outputs
# ------------------------------------------------------------

RETROSPECTIVE_OUTPUT_DIR = (
    OUTPUT_DIR
    / "tables"
)


RETROSPECTIVE_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


primary_pooled_results.to_csv(
    RETROSPECTIVE_OUTPUT_DIR
    / "retrospective_primary_logistic.csv",
    index=False,
)


compact_pooled_results.to_csv(
    RETROSPECTIVE_OUTPUT_DIR
    / "retrospective_compact_logistic.csv",
    index=False,
)


random_negative_placebo.to_csv(
    RETROSPECTIVE_OUTPUT_DIR
    / "random_negative_placebo.csv",
    index=False,
)


pure_negative_comparison.to_csv(
    RETROSPECTIVE_OUTPUT_DIR
    / "pure_negative_falsification.csv",
    index=False,
)


negative_pool_audit.to_csv(
    RETROSPECTIVE_OUTPUT_DIR
    / "negative_pool_audit.csv",
    index=False,
)


print(
    "Saved retrospective modeling outputs "
    "under outputs/tables/."
)

Saved retrospective modeling outputs under outputs/tables/.
